In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
import os

JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import pandas as pd

# Import your CRUD Python module
from animal_shelter import AnimalShelter


###########################
# Data Manipulation / Model
###########################

# Hardcoded username and password for Project Two
username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Retrieve all documents
df = pd.DataFrame.from_records(db.read({}))

# Drop MongoDB ObjectId column
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Grazioso Salvare logo
# If your logo filename is different, update this list with the correct filename.
possible_logo_files = [
    "Grazioso Salvare Logo.png",
    "grazioso salvare logo.png",
    "grazioso_salvare_logo.png",
    "my-image.png"
]

encoded_image = None

for logo_file in possible_logo_files:
    if os.path.exists(logo_file):
        encoded_image = base64.b64encode(open(logo_file, "rb").read())
        break


app.layout = html.Div([

    html.Center([
        html.H1("CS-340 Dashboard - Coden Polk"),

        # Logo will display if the image file is found
        html.Img(
            src="data:image/png;base64,{}".format(encoded_image.decode()) if encoded_image else "",
            style={
                "width": "250px",
                "height": "auto"
            }
        )
    ]),

    html.Hr(),

    html.H3("Rescue Type Filter"),

    dcc.RadioItems(
        id="filter-type",
        options=[
            {"label": "Reset", "value": "Reset"},
            {"label": "Water Rescue", "value": "Water Rescue"},
            {"label": "Mountain or Wilderness Rescue", "value": "Mountain or Wilderness Rescue"},
            {"label": "Disaster or Individual Tracking", "value": "Disaster or Individual Tracking"}
        ],
        value="Reset",
        labelStyle={
            "display": "inline-block",
            "margin-right": "20px"
        }
    ),

    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",

        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],

        data=df.to_dict("records"),

        # Required for map callback
        row_selectable="single",
        selected_rows=[0],

        # User-friendly table features
        page_size=10,
        sort_action="native",
        filter_action="native",

        style_table={
            "overflowX": "auto"
        },

        style_cell={
            "textAlign": "left",
            "minWidth": "100px",
            "width": "150px",
            "maxWidth": "200px",
            "whiteSpace": "normal"
        },

        style_header={
            "fontWeight": "bold"
        }
    ),

    html.Br(),
    html.Hr(),

    # Chart and map side-by-side
    html.Div(
        className="row",
        style={"display": "flex"},
        children=[
            html.Div(
                id="graph-id",
                className="col s12 m6",
                style={"width": "50%"}
            ),

            html.Div(
                id="map-id",
                className="col s12 m6",
                style={"width": "50%"}
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):

    if filter_type == "Water Rescue":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "Mountain or Wilderness Rescue":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "Disaster or Individual Tracking":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    else:
        query = {}

    filtered_df = pd.DataFrame.from_records(db.read(query))

    if "_id" in filtered_df.columns:
        filtered_df.drop(columns=["_id"], inplace=True)

    return filtered_df.to_dict("records")


# Display breed chart based on current data table
@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    if len(dff) == 0:
        return [html.H3("No data available for this filter.")]

    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names="breed",
                title="Preferred Animals by Breed"
            )
        )
    ]


# Highlight selected columns
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):

    if selected_columns is None:
        selected_columns = []

    return [{
        "if": {"column_id": i},
        "background_color": "#D2F3FF"
    } for i in selected_columns]


# Update map based on selected row
@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):

    if viewData is None:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    if len(dff) == 0:
        return [html.H3("No map data available.")]

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(
            style={
                "width": "1000px",
                "height": "500px"
            },
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),

                dl.Marker(
                    position=[
                        dff.iloc[row]["location_lat"],
                        dff.iloc[row]["location_long"]
                    ],
                    children=[
                        dl.Tooltip(dff.iloc[row]["breed"]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row]["name"])
                        ])
                    ]
                )
            ]
        )
    ]


# Run app
app.run_server(port=8056)

Dash app running on https://powdermorgan-archiveviking-3000.codio.io/proxy/8056/
